# **Project Name**    - PhonePe Pulse Data Visualization and Exploration


##### **Project Type**    - EDA / Machine Learning
##### **Contribution**    - Individual
##### **Team Member 1 -** Farheen Fathima


# **Project Summary -**

The PhonePe Pulse dataset provides a comprehensive look into digital payment trends across India. This project aims to extract, process, and analyze this massive dataset to extract meaningful insights. The data is initially scattered across hundreds of JSON files in a GitHub repository. We first built a custom script to clone the repository, parse the hierarchical directory structure, and aggregate the transaction, user, and insurance data into structured pandas DataFrames. These DataFrames were then migrated into a robust MySQL database for scalable querying.

In the EDA phase, we rigorously cleaned the data, standardizing state names and handling missing values. We deployed extensive visualizations, including choropleth maps and time-series line charts, to uncover geographical hot-spots, popular transaction types, and quarter-over-quarter growth patterns. The analysis revealed a massive surge in digital adoption, heavily dominated by peer-to-peer and merchant transactions, with significant penetration in both urban and rural districts.

For the Machine Learning phase, we engineered categorical features and trained powerful regression models (Random Forest, XGBoost, and Linear Regression) to predict transaction volumes. XGBoost yielded the best performance after hyperparameter tuning. Finally, the entire pipeline, including an interactive Streamlit dashboard for real-time data exploration, was packaged for deployment.


# **GitHub Link -**

https://github.com/farheenfathimaa/phonepe-project


# **Problem Statement**


PhonePe, one of India's leading fintech platforms, has open-sourced its transaction data through the PhonePe Pulse initiative. However, the raw data is distributed across thousands of nested JSON files, making it difficult to query, visualize, and extract business value directly. The problem is to engineer a complete, end-to-end data pipeline that automates the extraction of this data, cleans and structures it into a relational database, and builds an interactive dashboard to visualize the trends. Furthermore, we must perform deep Exploratory Data Analysis (EDA) and build predictive Machine Learning models to understand user behavior and forecast transaction amounts across different states and timeframes.


#### **Define Your Business Objective?**

To construct a scalable, automated data pipeline for extracting, transforming, and visualizing India's digital payment trends using the PhonePe Pulse dataset. The objective is to produce actionable intelligence regarding transaction behaviors, user growth, and market penetration to inform targeted expansion strategies.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import folium
import sqlalchemy
import config
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# Load Dataset
engine = sqlalchemy.create_engine(f'mysql+mysqlconnector://{config.DB_USER}:{config.DB_PASSWORD_ENCODED}@{config.DB_HOST}/{config.DB_NAME}')
tables = ['Aggregated_transaction', 'Aggregated_user', 'Aggregated_insurance', 'Map_transaction', 'Map_user', 'Map_insurance', 'Top_transaction', 'Top_user', 'Top_insurance']
dataframes = {}
with engine.connect() as conn:
    for table in tables:
        try:
            dataframes[table] = pd.read_sql(f"SELECT * FROM {table}", conn)
        except Exception as e:
            print(f"Error loading {table}: {e}")
df_trans = dataframes.get('Aggregated_transaction', pd.DataFrame())

### Dataset First View

In [ ]:
# Dataset First Look
display(df_trans.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
for name, df in dataframes.items():
    if not df.empty:
        print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

### Dataset Information

In [ ]:
# Dataset Info
df_trans.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
for name, df in dataframes.items():
    if not df.empty:
        print(f"{name} duplicates: {df.duplicated().sum()}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
for name, df in dataframes.items():
    if not df.empty:
        print(f"{name} missing values:
{df.isnull().sum()}
")

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10,6))
sns.heatmap(df_trans.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Aggregated_transaction)')
plt.show()

### What did you know about your dataset?

The dataset contains 9 distinct categories covering Aggregated, Map, and Top data for Transactions, Users, and Insurance. It spans multiple years and quarters, detailing the number and value of transactions, registered users, app opens, and device brands across all Indian states and districts. The data is clean with very few missing values, though standardizing state and district names was necessary.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print(df_trans.columns.tolist())

In [ ]:
# Dataset Describe
display(df_trans.describe())

### Variables Description

* **State/District**: Geographical location of the transaction or user.
* **Year/Quarter**: Temporal features indicating when the activity occurred.
* **Transaction_type**: The category of the transaction (e.g., Peer-to-peer, Merchant).
* **Transaction_count**: Total number of transactions.
* **Transaction_amount**: Total monetary value of the transactions.
* **Registered_users**: Total number of users registered on the platform.
* **Brand**: The smartphone brand used by the user.


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in df_trans.columns:
    print(f"{col} unique values: {df_trans[col].nunique()}")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
def clean_data(df):
    if df.empty: return df
    if 'State' in df.columns:
        df['State'] = df['State'].str.replace('-', ' ').str.title()
    df.dropna(inplace=True)
    return df
for name, df in dataframes.items():
    dataframes[name] = clean_data(df)
df_trans = dataframes.get('Aggregated_transaction', pd.DataFrame())

### What all manipulations have you done and insights you found?

1. Handled missing values by dropping sparse rows.
2. Standardized 'State' names by removing hyphens and capitalizing words (e.g., 'andaman-&-nicobar-islands' -> 'Andaman & Nicobar').
3. Consolidated the 9 different JSON structures into unified SQL tables.

**Insights:**
The data formatting revealed that PhonePe's transaction volume has grown exponentially year-over-year, with the majority of transactions clustered in highly populated states like Maharashtra and Karnataka.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
state_trans = df_trans.groupby('State')['Transaction_amount'].sum().reset_index()
state_trans = state_trans.sort_values(by='Transaction_amount', ascending=False)
fig = px.bar(state_trans, x='State', y='Transaction_amount', title='Total Transaction Amount by State')
fig.show()

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 2

In [ ]:
# Chart - 2 visualization code
type_trans = df_trans.groupby('Transaction_type')['Transaction_amount'].sum().reset_index()
fig = px.pie(type_trans, values='Transaction_amount', names='Transaction_type', title='Transaction Types Distribution')
fig.show()

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 3

In [ ]:
# Chart - 3 visualization code
trend = df_trans.groupby(['Year', 'Quarter'])['Transaction_amount'].sum().reset_index()
trend['Period'] = trend['Year'].astype(str) + " Q" + trend['Quarter'].astype(str)
fig = px.line(trend, x='Period', y='Transaction_amount', markers=True, title='Transaction Trend Over Time')
fig.show()

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 4

In [ ]:
# Chart - 4 visualization code
geojson_url = "https://gist.githubusercontent.com/jbrobst/56c13bbbf9d97d187fea01ca62ea5112/raw/e388c4cae20aa53cb5090210a42ebb9b765c0a36/india_states.geojson"
fig = px.choropleth(
    state_trans,
    geojson=geojson_url,
    featureidkey='properties.ST_NM',
    locations='State',
    color='Transaction_amount',
    color_continuous_scale='Viridis',
    title='Choropleth Map: Transaction Amount by State'
)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 5

In [ ]:
# Chart - 5 visualization code
df_map_trans = dataframes.get('Map_transaction', pd.DataFrame())
if not df_map_trans.empty:
    dist_trans = df_map_trans.groupby('District')['Transaction_count'].sum().reset_index()
    dist_trans = dist_trans.sort_values(by='Transaction_count', ascending=False).head(10)
    fig = px.bar(dist_trans, x='District', y='Transaction_count', title='Top 10 Districts by Transaction Count')
    fig.show()

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 6

In [ ]:
# Chart - 6 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 7

In [ ]:
# Chart - 7 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 8

In [ ]:
# Chart - 8 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 9

In [ ]:
# Chart - 9 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 10

In [ ]:
# Chart - 10 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 11

In [ ]:
# Chart - 11 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 12

In [ ]:
# Chart - 12 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 13

In [ ]:
# Chart - 13 visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding which states and categories drive the most volume allows PhonePe to allocate marketing budgets more effectively and target underdeveloped regions with specific campaigns.


#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

##### 1. Why did you pick the specific chart?

This chart was selected to clearly visualize the distribution and comparison of the target metric across different categories, allowing for easy identification of the highest and lowest performing segments.


##### 2. What is/are the insight(s) found from the chart?

The insight is that digital payments are highly skewed towards specific major states (like Maharashtra) and transaction types (like Peer-to-peer), indicating clear market leaders.


## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

1. **Targeted Expansion**: Focus marketing efforts in Tier-2 and Tier-3 cities within states that currently show low transaction volumes.
2. **Merchant Incentives**: Since Merchant payments form a large chunk, incentivizing more local businesses to adopt PhonePe will drive up transaction counts.
3. **Insurance Cross-Selling**: The insurance segment is relatively small compared to core transactions. PhonePe should cross-sell insurance products to its highly engaged peer-to-peer users.


# **Conclusion**

The EDA phase revealed massive year-over-year growth in PhonePe's digital adoption across all metrics. By visualizing the data through choropleth maps and categorical charts, we successfully identified market leaders (Maharashtra, Karnataka) and key behaviors (Peer-to-peer dominance). This concludes the Exploratory Data Analysis phase, preparing the structured data for predictive modeling.


### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***